# 01 Genie Agent の作成（API）

`00_セットアップ` で準備したテーブルと Volume を使って、**Genie Agent を API で作成**します。画面で1つずつ設定する内容（テーブル・Volume・指示・サンプルの質問・ベンチマーク）を、まとめて JSON で渡します。

比べるために、Agent を2つ作ります。

| Agent | テーブル4つ | Volume `docs`（文書34ファイル） |
|---|:---:|:---:|
| `[Volumeあり] ブリックス電機 品質保証部 分析エージェント` | ✅ | ✅ |
| `[Volumeなし] ブリックス電機 品質保証部 分析エージェント` | ✅ | — |

**準備するもの**
- `00_セットアップ` の実行が終わっていること
- サーバーレスのコンピュート（右上の［接続］から選ぶ）
- Pro またはサーバーレスの SQL ウェアハウスへの `CAN USE`（Genie Agent が SQL を実行する）

上のウィジェットで **カタログ名** と **スキーマ名**（`00_セットアップ` と同じ）を指定してから、［すべてを実行］を押してください（1分以内）。ウェアハウス ID は空欄のままでかまいません（使えるウェアハウスを自動で選びます）。

In [0]:
dbutils.widgets.text("catalog", "", "1 カタログ名")
dbutils.widgets.text("schema", "bricks_electronics", "2 スキーマ名")
dbutils.widgets.text("warehouse_id", "", "3 ウェアハウス ID（空欄なら自動）")
dbutils.widgets.text("agent_name", "ブリックス電機 品質保証部 分析エージェント", "4 Agent の名前")

In [0]:
import json
import os
import uuid

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()   # ノートブックを実行している人の権限で API を呼ぶ

catalog = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
warehouse_id = dbutils.widgets.get("warehouse_id").strip()
agent_name = dbutils.widgets.get("agent_name").strip()
assert catalog and schema_name and agent_name, "カタログ名・スキーマ名・Agent の名前を指定してください"

me = w.current_user.me().user_name
# Agent を置くフォルダ（このノートブックと同じフォルダ）
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_path = os.path.dirname(notebook_path).removeprefix("/Workspace")
host = w.config.host.rstrip("/")

# ウェアハウス ID が空欄なら、使える SQL ウェアハウスを選ぶ（サーバーレスを優先）
if not warehouse_id:
    whs = [wh for wh in w.warehouses.list() if wh.warehouse_type and wh.warehouse_type.value == "PRO"]
    assert whs, "Pro またはサーバーレスの SQL ウェアハウスが見つかりません。warehouse_id を指定してください"
    # 起動中 → サーバーレス → 名前の順で選ぶ
    whs.sort(key=lambda wh: (wh.state is None or wh.state.value != "RUNNING", not wh.enable_serverless_compute, wh.name))
    warehouse_id = whs[0].id
    print(f"ウェアハウス（自動で選択）: {whs[0].name} ({warehouse_id})")
else:
    print(f"ウェアハウス: {w.warehouses.get(warehouse_id).name} ({warehouse_id})")

c = f"{catalog}.{schema_name}"
TABLES = [f"{c}.{t}" for t in ["products", "production_lots", "sales_monthly", "repair_requests"]]
DOCS_VOLUME = f"/Volumes/{catalog}/{schema_name}/docs"
print(f"テーブル : {', '.join(TABLES)}")
print(f"Volume   : {DOCS_VOLUME}")
print(f"作成先   : {parent_path}")

## 1. Agent の設定を組み立てる

Agent の設定は、`serialized_space` という **JSON の文字列**で渡します。画面の設定との対応は次のとおりです。

| 画面（［設定する］） | `serialized_space` のキー | このノートブックで入れるもの |
|---|---|---|
| ［ソース］テーブル | `data_sources.tables` | テーブル4つ（`identifier` の**昇順に並べる**） |
| ［ソース］ボリューム | `data_sources.volumes` | Volume `docs` と、その説明（`[Volumeあり]` だけ） |
| ［指示］一般的な指示 | `instructions.text_instructions` | 業務の説明（`02_シナリオと想定QA` の 4-3 と同じ文章） |
| ［概要］よくある質問 | `config.sample_questions` | 分析の質問 |
| ［ベンチマーク］ | `benchmarks.questions` | 答えを確かめるテスト用の質問と、正解の SQL |

名前（`title`）・説明（`description`）・ウェアハウス（`warehouse_id`）・置き場所（`parent_path`）は、`serialized_space` の外に書きます。

In [0]:
INSTRUCTIONS = """
# あなたの役割
あなたは、家電メーカー「ブリックス電機」の品質保証部の分析エージェントです。修理受付のデータ（テーブル）と社内文書（Volume）を組み合わせて、修理の傾向の監視、不具合の原因の調査、対策（無償点検・設計変更など）とその費用の検討を手伝います。

# データの使い分け
- 件数・率・金額は、必ずテーブルを集計して答える。
- 原因や背景（部品・工程の変更、過去の不具合と対策、使いかたの説明、費用の単価）は、文書（docs Volume）から探す。
  - quality_meetings/: 品質保証部の月次の品質会議の資料。部品・工程の変更（変更点管理）、対策の進捗、決定事項
  - service_bulletins/: サービス技術情報。既知の不具合と、受付時・修理時の対応
  - manuals/: 製品ごとの取扱説明書。使いかた、お手入れ、トラブル対処の表、エラー表示の一覧
  - policies/: 保証規定・修理料金表。保証の条件、技術料・部品代・送料
- 答えには、根拠にしたテーブル（集計の条件）と文書の名前を示す。

# 指標の定義
- 受付率 = 修理受付の件数 ÷ 販売台数 × 1,000（販売1,000台あたり）。製品の比較や増減の判断は、件数ではなく受付率で行う（扇風機・ヒーター・加湿器は季節で販売台数が大きく変わるため）。sales_monthly は地域別なので、型番と月で合計してから割る。
- 「実際には修理が不要だったもの」は、result_category が「修理不要」（電話・メールの案内で解決）と「異常なし」（修理を手配したが、点検の結果、故障ではなかった）の受付。
- ロットごとの不具合の多さは、ロットごとの受付件数 ÷ production_lots.quantity（製造台数）で比べる。
- 費用の目安は、修理料金表の単価（技術料・部品代・送料）と、過去の無償交換の実績（実施率・1台あたりの費用・告知費用）を根拠にする。

# 注意
- operator_memo には、お客様の氏名・電話番号・メールアドレス・住所が含まれる。回答に個人情報を出さず、件数や傾向に要約する。
- 型番が似ている製品（RC-500 と RC-550）を取り違えない。
"""

SAMPLE_QUESTIONS = [
    "この半年で修理の受付が増えている製品を特定し、原因を分析して。さらに修理対象の製品に関して、保証対象の判断基準と実際に取られた保証対応について教えて。",
]

# ベンチマーク: 質問と、正解の結果を返す SQL（{c} はカタログ.スキーマに置き換える）
BENCHMARKS = [
    (
        "2026年4〜9月の製品別の受付率（販売1,000台あたり）を、高い順に出して",
        """
SELECT r.model_no, r.requests, s.units_sold,
       ROUND(r.requests / s.units_sold * 1000, 1) AS requests_per_1000_units
FROM (SELECT model_no, COUNT(*) AS requests FROM {c}.repair_requests
      WHERE received_at >= '2026-04-01' AND received_at < '2026-10-01' GROUP BY model_no) r
JOIN (SELECT model_no, SUM(units_sold) AS units_sold FROM {c}.sales_monthly
      WHERE sales_month >= '2026-04-01' AND sales_month < '2026-10-01' GROUP BY model_no) s
  ON r.model_no = s.model_no
ORDER BY requests_per_1000_units DESC
""",
    ),
    (
        "RC-500 の2026年の月別の修理受付件数を出して",
        """
SELECT DATE_TRUNC('MONTH', received_at) AS month, COUNT(*) AS requests
FROM {c}.repair_requests
WHERE model_no = 'RC-500' AND received_at >= '2026-01-01' AND received_at < '2027-01-01'
GROUP BY 1 ORDER BY 1
""",
    ),
    (
        "RC-500 のロット別の受付件数と、製造1,000台あたりの受付件数を、多い順に上位10ロット出して",
        """
SELECT l.lot_no, l.factory, l.production_month, l.quantity, COUNT(r.request_id) AS requests,
       ROUND(COUNT(r.request_id) / l.quantity * 1000, 1) AS requests_per_1000_units
FROM {c}.production_lots l
LEFT JOIN {c}.repair_requests r ON r.lot_no = l.lot_no
WHERE l.model_no = 'RC-500'
GROUP BY l.lot_no, l.factory, l.production_month, l.quantity
ORDER BY requests_per_1000_units DESC
LIMIT 10
""",
    ),
    (
        "対応結果ごとの修理受付件数を出して",
        """
SELECT result_category, COUNT(*) AS requests
FROM {c}.repair_requests
GROUP BY result_category ORDER BY requests DESC
""",
    ),
    (
        "受付チャネル別に、修理が不要だった受付（修理不要・異常なし）の割合を出して",
        """
SELECT channel, COUNT(*) AS requests,
       SUM(CASE WHEN result_category IN ('修理不要', '異常なし') THEN 1 ELSE 0 END) AS no_repair_needed,
       ROUND(SUM(CASE WHEN result_category IN ('修理不要', '異常なし') THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS no_repair_needed_pct
FROM {c}.repair_requests
GROUP BY channel ORDER BY channel
""",
    ),
]

VOLUME_DESCRIPTION = (
    "ブリックス電機の社内文書（PDF・PPTX）。manuals/ = 取扱説明書（製品ごとに1本）、"
    "quality_meetings/ = 品質保証部の月次の品質会議の資料、service_bulletins/ = サービス技術情報"
    "（既知の不具合と対処の通達）、policies/ = 保証規定・修理料金表"
)


def new_id():
    return uuid.uuid4().hex   # 32文字の16進数


def lines(text):
    """複数行の文字列を、行ごとの配列にする（serialized_space の文字列は配列で書く）."""
    return text.strip("\n").splitlines(keepends=True)


def build_space(with_volume):
    data_sources = {"tables": [{"identifier": t} for t in sorted(TABLES)]}
    if with_volume:
        data_sources["volumes"] = [{"path": DOCS_VOLUME, "description": [VOLUME_DESCRIPTION]}]
    return {
        "version": 2,
        "config": {"sample_questions": [{"id": new_id(), "question": [q]} for q in SAMPLE_QUESTIONS]},
        "data_sources": data_sources,
        "instructions": {"text_instructions": [{"id": new_id(), "content": lines(INSTRUCTIONS)}]},
        "benchmarks": {"questions": [
            {"id": new_id(), "question": [q], "answer": [{"format": "SQL", "content": lines(sql.format(c=c))}]}
            for q, sql in BENCHMARKS
        ]},
    }


space = build_space(with_volume=True)
print(json.dumps(space, ensure_ascii=False, indent=2)[:1500], "…")

## 2. API で Agent を作る

`POST /api/2.0/genie/spaces` に、名前・ウェアハウス・置き場所と `serialized_space` を送ると、Agent が作られます（レスポンスの `space_id` が Agent の ID）。

- 同じフォルダに同じ名前の Agent があると作成できないため、**同名の Agent があれば、`PATCH /api/2.0/genie/spaces/<space_id>` で設定を上書き**します（何度実行しても、Agent は2つのまま）。会話の履歴やベンチマークの実行記録は消えません。

In [0]:
MARK = "（01_Genie Agent の作成（API）で作成）"
MARK_KEY = "Genie Agent の作成（API）で作成"   # 名前を変える前に作った Agent も見分ける
DESCRIPTION = ("修理受付のデータ（テーブル）と社内文書（取扱説明書・品質会議の資料・サービス技術情報・保証規定）を"
               "横断して、不具合の原因と対策を分析する品質保証部のエージェント")


def find_agent(title):
    """作成先のフォルダ（parent_path）にある、同じ名前の Agent を探す."""
    page_token = None
    while True:
        res = w.api_client.do("GET", "/api/2.0/genie/spaces",
                              query={"page_size": 100, **({"page_token": page_token} if page_token else {})})
        for s in res.get("spaces", []):
            if s.get("title") == title:
                detail = w.api_client.do("GET", f"/api/2.0/genie/spaces/{s['space_id']}")
                if detail.get("parent_path", "").removeprefix("/Workspace") == parent_path:
                    return detail
        page_token = res.get("next_page_token")
        if not page_token:
            return None


def create_or_update(title, description, with_volume):
    body = {
        "title": title,
        "description": description,
        "warehouse_id": warehouse_id,
        "serialized_space": json.dumps(build_space(with_volume), ensure_ascii=False),
    }
    found = find_agent(title)
    if found is None:
        res = w.api_client.do("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": parent_path})
        action = "作成しました"
    elif MARK_KEY in (found.get("description") or ""):
        res = w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{found['space_id']}", body=body)
        action = "更新しました"
    else:
        res = w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{found['space_id']}", body=body)
        action = "上書きしました（既存の Agent）"
    return res["space_id"], action


agents = {}
for label, with_volume in [("Volumeあり", True), ("Volumeなし", False)]:
    title = f"[{label}] {agent_name}"
    desc = DESCRIPTION + ("" if with_volume else "（比較用: 文書の Volume なし）") + MARK
    space_id, action = create_or_update(title, desc, with_volume)
    agents[label] = space_id
    print(f"✅ {action}: {title}")
    print(f"   {host}/genie/rooms/{space_id}")

## 3. 作った Agent を確かめる

API で Agent の設定を取り出し（`GET /api/2.0/genie/spaces/<space_id>?include_serialized_space=true`）、テーブル・Volume・指示・ベンチマークが入っているかを確かめます。

In [0]:
for label, space_id in agents.items():
    res = w.api_client.do("GET", f"/api/2.0/genie/spaces/{space_id}", query={"include_serialized_space": "true"})
    s = json.loads(res["serialized_space"])
    ds = s.get("data_sources", {})
    n_chars = sum(len("".join(t["content"])) for t in s.get("instructions", {}).get("text_instructions", []))
    print(res["title"])
    print(f"   テーブル       : {len(ds.get('tables', []))} 個")
    print(f"   Volume         : {', '.join(v['path'] for v in ds.get('volumes', [])) or 'なし'}")
    print(f"   指示           : {n_chars:,} 文字")
    print(f"   サンプルの質問 : {len(s.get('config', {}).get('sample_questions', []))} 個")
    print(f"   ベンチマーク   : {len(s.get('benchmarks', {}).get('questions', []))} 問")

print("\n🎉 Agent を2つ作りました。上のリンクから [Volumeあり] の Agent を開いてください")

## 4. 次にすること

1. 上のリンクから **`[Volumeあり]`** の Agent を開き、［設定する］→［ソース］で `docs` の **コンテンツ検索のステータスが「同期完了」、処理済みファイルが 34/34** になるまで待ちます（Volume を追加すると、インデックスが自動で作られます。最初は数分かかります）。
2. ［指示］［例］［ベンチマーク］に、API で入れた設定が入っていることを確かめます。
3. `02_シナリオと想定QA` の **Step 3（データの確認）・Step 4（分析）** に進み、`[Volumeあり]` と `[Volumeなし]` に同じ質問を送って、答えを比べます。

> 💡 同じ内容は CLI でも作れます: `databricks genie create-space <ウェアハウスID> '<serialized_space>' --title ... --parent-path ...`（[Genie コマンド](https://docs.databricks.com/aws/ja/dev-tools/cli/reference/genie-commands)）